## Module 1 Homework (2026 cohort)

In this homework, we're going to download finance data from various sources and make simple calculations or analysis.

### Question 1: [Index] S&P 500 Stocks Added to the Index

**Which year had the highest number of additions?**

Using the list of S&P 500 companies from Wikipedia's [S&P 500 companies page](https://en.wikipedia.org/wiki/List_of_S%26P_500_companies), download the data including the year each company was added to the index.

Hint: you can use [pandas.read_html](https://pandas.pydata.org/docs/reference/api/pandas.read_html.html) to scrape the data into a DataFrame. You may need to add headers to make it work:

In [1]:
import io
import requests
import pandas as pd

url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                  '(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

tables = pd.read_html(io.StringIO(response.text))
print(f"pandas version: {pd.__version__}")
print(f"tables found: {len(tables)}")
for i, t in enumerate(tables):
    print(i, t.shape, list(t.columns)[:6])

pandas version: 3.0.5
tables found: 2
0 (503, 8) ['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry', 'Headquarters Location', 'Date added']
1 (11, 2) ['vteS&P 500 companies', 'vteS&P 500 companies.1']


In [2]:
df = tables[0].copy()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   Symbol                 503 non-null    str  
 1   Security               503 non-null    str  
 2   GICS Sector            503 non-null    str  
 3   GICS Sub-Industry      503 non-null    str  
 4   Headquarters Location  503 non-null    str  
 5   Date added             503 non-null    str  
 6   CIK                    503 non-null    int64
 7   Founded                503 non-null    str  
dtypes: int64(1), str(7)
memory usage: 75.8 KB


In [3]:
df.head()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


In [4]:
df['Year added'] = pd.to_datetime(df['Date added'], errors='coerce').dt.year
print(f"\nUnparsed dates: {df['Year added'].isna().sum()}")


Unparsed dates: 0


### Step 2: Count additions per year

`Date added` parsed cleanly (503/503), so we can take the year directly.
Counting by year gives the number of tickers added in each calendar year.

Two caveats to keep in mind when reading the result:
- **1957-03-04** is not a normal addition event. It is the date the index
  took its modern 500-company form, so ~50 tickers share it. It reflects
  index construction, not annual turnover.
- **2026 is incomplete** — the data was pulled in August 2026, so that year
  covers roughly eight months. Q1 asks for the highest *full* year, which
  excludes 2026 from the answer.

In [5]:
additions_per_year = df['Year added'].value_counts().sort_index()
additions_per_year.tail(15)

Year added
2012    14
2013     9
2014     8
2015    14
2016    21
2017    22
2018    13
2019    21
2020    10
2021    10
2022    15
2023    15
2024    16
2025    18
2026    13
Name: count, dtype: int64

### Step 3: Highest-addition year since 2020

Filter to 2020 onward and exclude 2026, which is incomplete (data pulled
August 2026). Take the year with the maximum count.

Note: this table lists current constituents, so companies added and later
removed are missing. Counts are lower bounds, more so for earlier years.

In [6]:
full_years = additions_per_year.loc[2020:2025]

answer_year = full_years.idxmax()
answer_count = full_years.max()

print(full_years)
print(f"\nAnswer: {answer_year} ({answer_count} additions)")

Year added
2020    10
2021    10
2022    15
2023    15
2024    16
2025    18
Name: count, dtype: int64

Answer: 2025 (18 additions)


### Additional: constituents in the index more than 20 years

Count current constituents added more than 20 years before the pull date.

In [7]:
print(f"Tickers in index >20 years: {(df['Year added'] < 2006).sum()} of {len(df)}")

Tickers in index >20 years: 218 of 503


### Question 2. [Macro] Indexes YTD (as of 21 August 2026)

**How many indexes (out of 10) have better year-to-date returns than the US (S&P 500) as of August 21, 2026?**

Using Yahoo Finance World Indices data, compare the year-to-date (YTD) performance (1 January-1 August 2026, use CLOSE price for the day) of major stock market indexes for the following countries:
* United States - S&P 500 (^GSPC)
* China - Shanghai Composite (000001.SS)
* Hong Kong - HANG SENG INDEX (^HSI)	
* Australia - S&P/ASX 200 (^AXJO)
* India - Nifty 50 (^NSEI)
* Canada - S&P/TSX Composite (^GSPTSE)
* Germany - DAX (^GDAXI)
* United Kingdom - FTSE 100 (^FTSE)
* Japan - Nikkei 225 (^N225)
* Mexico - IPC Mexico (^MXX)
* Brazil - Ibovespa (^BVSP)

*Hint*: use start_date='2026-01-01' and end_date='2026-08-21' when downloading daily data in yfinance

Context: 
> [Global Valuations: Who's Cheap, Who's Not?](https://simplywall.st/article/beyond-the-us-global-markets-after-yet-another-tariff-update) article suggests "Other regions may be growing faster than the US and you need to diversify."

Reference: Yahoo Finance World Indices - https://finance.yahoo.com/world-indices/

In [8]:
import yfinance as yf
import pandas as pd

indexes = {
    '^GSPC':     'US - S&P 500',
    '000001.SS': 'China - Shanghai Composite',
    '^HSI':      'Hong Kong - Hang Seng',
    '^AXJO':     'Australia - ASX 200',
    '^NSEI':     'India - Nifty 50',
    '^GSPTSE':   'Canada - TSX Composite',
    '^GDAXI':    'Germany - DAX',
    '^FTSE':     'UK - FTSE 100',
    '^N225':     'Japan - Nikkei 225',
    '^MXX':      'Mexico - IPC',
    '^BVSP':     'Brazil - Ibovespa',
}

raw = yf.download(
    list(indexes.keys()),
    start='2026-01-01',
    end='2026-08-22',
    interval='1d',
    auto_adjust=False,
    progress=False,
)

close = raw['Close']
print(close.shape)
print(close.index.min(), '→', close.index.max())
print()
print(close.notna().sum())

(167, 11)
2026-01-01 00:00:00 → 2026-08-21 00:00:00

Ticker
000001.SS    154
^AXJO        162
^BVSP        160
^FTSE        162
^GDAXI       163
^GSPC        160
^GSPTSE      161
^HSI         156
^MXX         161
^N225        155
^NSEI        157
dtype: int64


### YTD return per index

Each market has its own holiday calendar, so the combined frame has NaNs.
Take the first and last *available* close per column rather than positional
first/last rows.

    YTD return = last_close / first_close - 1

In [9]:
first = close.apply(lambda s: s.loc[s.first_valid_index()])
last  = close.apply(lambda s: s.loc[s.last_valid_index()])

ytd = pd.DataFrame({
    'index_name': pd.Series(indexes),
    'first_date': close.apply(lambda s: s.first_valid_index()),
    'last_date':  close.apply(lambda s: s.last_valid_index()),
    'first_close': first,
    'last_close': last,
    'ytd_return_pct': (last / first - 1) * 100,
}).sort_values('ytd_return_pct', ascending=False)

display(ytd.round(2))

/var/folders/qg/s6tmww6s0zv7vs41wx8t_07m0000gn/T/ipykernel_1884/1903392894.py:13: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(ytd.round(2))


,index_name,first_date,last_date,first_close,last_close,ytd_return_pct
^N225,Japan - Nikkei 225,2026-01-05,2026-08-21,51832.80,66016.36,27.36
^GSPTSE,Canada - TSX Composite,2026-01-02,2026-08-21,31883.40,36620.20,14.86
^GSPC,US - S&P 500,2026-01-02,2026-08-21,6858.47,7674.37,11.90
^FTSE,UK - FTSE 100,2026-01-02,2026-08-21,9951.10,10816.60,8.70
^BVSP,Brazil - Ibovespa,2026-01-02,2026-08-21,160539.00,171032.00,6.54
^GDAXI,Germany - DAX,2026-01-02,2026-08-21,24539.34,26136.56,6.51
^AXJO,Australia - ASX 200,2026-01-02,2026-08-21,8727.80,9058.90,3.79
^MXX,Mexico - IPC,2026-01-02,2026-08-21,64141.36,65729.18,2.48
^HSI,Hong Kong - Hang Seng,2026-01-02,2026-08-21,26338.47,26009.46,-1.25
000001.SS,China - Shanghai Composite,2026-01-05,2026-08-21,4023.42,3905.20,-2.94


### Answer

Count indexes with a YTD return above the S&P 500.

In [10]:
sp500 = ytd.loc['^GSPC', 'ytd_return_pct']
better = ytd[ytd['ytd_return_pct'] > sp500]

print(f"S&P 500 YTD: {sp500:.2f}%")
print(f"Indexes beating it: {len(better)}\n")
print(better[['index_name', 'ytd_return_pct']].round(2).to_string(index=False))

S&P 500 YTD: 11.90%
Indexes beating it: 2

            index_name  ytd_return_pct
    Japan - Nikkei 225           27.36
Canada - TSX Composite           14.86


### Additional: 3, 5 and 10-year comparison

Same 11 indexes, trailing horizons ending 2026-08-21. Anchor dates
(2023-08-21, 2021-08-21, 2016-08-21) may fall on holidays or weekends, so we
take the last available close on or before each anchor.

In [11]:
hist = yf.download(
    list(indexes.keys()),
    start='2016-08-01',
    end='2026-08-22',
    interval='1d',
    auto_adjust=False,
    progress=False,
)['Close']

print(hist.shape)
print(hist.apply(lambda s: s.first_valid_index()))

(2624, 11)
Ticker
000001.SS   2016-08-01
^AXJO       2016-08-01
^BVSP       2016-08-01
^FTSE       2016-08-01
^GDAXI      2016-08-01
^GSPC       2016-08-01
^GSPTSE     2016-08-02
^HSI        2016-08-01
^MXX        2016-08-01
^N225       2016-08-01
^NSEI       2016-08-01
dtype: datetime64[s]


Helper: close on the last trading day at or before a given date, per index.
Report total return over each horizon plus the annualized (CAGR) equivalent,
since 3/5/10-year totals aren't directly comparable to each other.

In [12]:
def close_asof(date):
    """Last available close on or before `date`, per index."""
    window = hist.loc[:date]
    return window.apply(lambda s: s.loc[s.last_valid_index()])

end_close = close_asof('2026-08-21')
horizons = {3: '2023-08-21', 5: '2021-08-21', 10: '2016-08-21'}

perf = pd.DataFrame({
    f'{years}y_pct': (end_close / close_asof(start) - 1) * 100
    for years, start in horizons.items()
})
perf.insert(0, 'index_name', pd.Series(indexes))
perf['ytd_pct'] = ytd['ytd_return_pct']

display(perf.sort_values('10y_pct', ascending=False).round(2))

,index_name,3y_pct,5y_pct,10y_pct,ytd_pct
Ticker,,,,,
^N225,Japan - Nikkei 225,109.14,144.39,298.99,27.36
^GSPC,US - S&P 500,74.43,72.78,251.41,11.90
^BVSP,Brazil - Ibovespa,49.47,44.88,189.40,6.54
^NSEI,India - Nifty 50,25.05,47.42,179.82,-7.25
^GSPTSE,Canada - TSX Composite,85.09,80.05,149.33,14.86
^GDAXI,Germany - DAX,67.51,65.34,147.87,6.51
^AXJO,Australia - ASX 200,27.31,21.42,63.91,3.79
^FTSE,UK - FTSE 100,49.03,52.61,57.70,8.70
^MXX,Mexico - IPC,23.76,27.84,36.09,2.48


In [13]:
for years in horizons:
    col = f'{years}y_pct'
    bench = perf.loc['^GSPC', col]
    winners = perf[perf[col] > bench].sort_values(col, ascending=False)
    print(f"--- {years}Y | S&P 500: {bench:.1f}% | beating it: {len(winners)} ---")
    print(winners[['index_name', col]].round(1).to_string(index=False))
    print()

--- 3Y | S&P 500: 74.4% | beating it: 2 ---
            index_name  3y_pct
    Japan - Nikkei 225   109.1
Canada - TSX Composite    85.1

--- 5Y | S&P 500: 72.8% | beating it: 2 ---
            index_name  5y_pct
    Japan - Nikkei 225   144.4
Canada - TSX Composite    80.0

--- 10Y | S&P 500: 251.4% | beating it: 1 ---
        index_name  10y_pct
Japan - Nikkei 225    299.0



### Answer (Additional)

Indexes beating the S&P 500, out of 10:
- YTD 2026: 2 (Japan, Canada)
- 3 years:  2 (Japan, Canada)
- 5 years:  2 (Japan, Canada)
- 10 years: 1 (Japan)

Yes, the same trend holds. The S&P 500 outperforms 8 of 10 indexes over
every period, and only Japan beats it at all four horizons.

### Question 3. [Index] S&P 500 Market Corrections Analysis


**Calculate the median drawdown (in %) of significant market corrections in the S&P 500 index.**

For this task, define a correction as an event when a stock index goes down by **more than 5%** from the closest all-time high maximum.

Steps:
1. Download S&P 500 historical data (1950-present) using yfinance
2. Identify all-time high points (where price exceeds all previous prices)
3. For each pair of consecutive all-time highs, find the minimum price in between
4. Calculate drawdown percentages: (high - low) / high × 100
5. Filter for corrections with at least 5% drawdown
6. Calculate the duration in days for each correction period
7. Determine the 25th, 50th (median), and 75th percentiles for correction durations and drawdowns

*Context:* 
> * Investors often wonder about the typical length of market corrections when deciding "when to buy the dip" ([Reddit discussion](https://www.reddit.com/r/investing/comments/1jrqnte/when_are_you_buying_the_dip/?rdt=64135)).

> * [A Wealth of Common Sense - How Often Should You Expect a Stock Market Correction?](https://awealthofcommonsense.com/2022/01/how-often-should-you-expect-a-stock-market-correction/)

*Hint (use this data to compare with your results)*: Here is the list of top 10 largest corrections by drawdown:
* 2007-10-09 to 2009-03-09: 56.8% drawdown over 517 days
* 2000-03-24 to 2002-10-09: 49.1% drawdown over 929 days
* 1973-01-11 to 1974-10-03: 48.2% drawdown over 630 days
* 1968-11-29 to 1970-05-26: 36.1% drawdown over 543 days
* 2020-02-19 to 2020-03-23: 33.9% drawdown over 33 days
* 1987-08-25 to 1987-12-04: 33.5% drawdown over 101 days
* 1961-12-12 to 1962-06-26: 28.0% drawdown over 196 days
* 1980-11-28 to 1982-08-12: 27.1% drawdown over 622 days
* 2022-01-03 to 2022-10-12: 25.4% drawdown over 282 days
* 1966-02-09 to 1966-10-07: 22.2% drawdown over 240 days

In [14]:
import yfinance as yf
import pandas as pd

raw = yf.download(
    '^GSPC',
    start='1950-01-01',
    end='2026-08-22',
    interval='1d',
    auto_adjust=False,
    progress=False,
)

close = raw['Close']['^GSPC']

print(close.shape)
print(close.index.min(), '→', close.index.max())
print(f"NaNs: {close.isna().sum()}")
close.head(3)

(19281,)
1950-01-03 00:00:00 → 2026-08-21 00:00:00
NaNs: 0


Date
1950-01-03    16.66
1950-01-04    16.85
1950-01-05    16.93
Name: ^GSPC, dtype: float64

### Identify all-time highs and the drawdown that follows each

A day is an all-time high when its close equals the running maximum of the
series. Numbering the ATHs with a cumulative sum assigns every day to the
"epoch" that began at its most recent ATH, so each group runs from one ATH
up to (but not including) the next.

Within an epoch the ATH day is by construction the maximum — any later day
exceeding it would itself be an ATH — so `idxmax`/`max` give the peak and
`idxmin`/`min` give the trough in between.

In [15]:
is_ath = close == close.cummax()
epoch = is_ath.cumsum()

g = close.groupby(epoch)

corrections = pd.DataFrame({
    'peak_date':   g.idxmax(),
    'peak':        g.max(),
    'trough_date': g.idxmin(),
    'trough':      g.min(),
})

corrections['drawdown_pct']  = (corrections['peak'] - corrections['trough']) / corrections['peak'] * 100
corrections['duration_days'] = (corrections['trough_date'] - corrections['peak_date']).dt.days

print(f"All-time highs: {is_ath.sum()}")
print(f"Peak-to-trough periods: {len(corrections)}")
corrections.tail(3)

All-time highs: 1537
Peak-to-trough periods: 1537


,peak_date,peak,trough_date,trough,drawdown_pct,duration_days
^GSPC,,,,,,
1535,2026-08-04,7736.520020,2026-08-06,7709.959961,0.343308,2
1536,2026-08-07,7757.640137,2026-08-11,7728.200195,0.379496,4
1537,2026-08-13,7798.990234,2026-08-20,7641.160156,2.023725,7


### Filter to corrections and verify against the assignment's hint table

Keep periods with a drawdown greater than 5%. Sorting by drawdown should
reproduce the top 10 in the assignment hint (2007-10-09 to 2009-03-09 at
56.8% over 517 days, and so on).

In [16]:
corr = corrections[corrections['drawdown_pct'] > 5].copy()

print(f"Corrections >5%: {len(corr)}\n")

top10 = corr.nlargest(10, 'drawdown_pct')
print(
    top10.assign(
        peak_date=top10['peak_date'].dt.date,
        trough_date=top10['trough_date'].dt.date,
    )[['peak_date', 'trough_date', 'drawdown_pct', 'duration_days']]
    .round(1)
    .to_string(index=False)
)

Corrections >5%: 74

 peak_date trough_date  drawdown_pct  duration_days
2007-10-09  2009-03-09          56.8            517
2000-03-24  2002-10-09          49.1            929
1973-01-11  1974-10-03          48.2            630
1968-11-29  1970-05-26          36.1            543
2020-02-19  2020-03-23          33.9             33
1987-08-25  1987-12-04          33.5            101
1961-12-12  1962-06-26          28.0            196
1980-11-28  1982-08-12          27.1            622
2022-01-03  2022-10-12          25.4            282
1966-02-09  1966-10-07          22.2            240


### Answer: percentiles of drawdown and duration

Report the 25th, 50th (median) and 75th percentiles for both the drawdown
magnitude and the peak-to-trough duration across the 74 corrections.

In [17]:
pct = corr[['drawdown_pct', 'duration_days']].quantile([0.25, 0.50, 0.75])
display(pct.round(2))

print(f"\nMedian drawdown: {corr['drawdown_pct'].median():.2f}%")
print(f"Median duration: {corr['duration_days'].median():.1f} days")

,drawdown_pct,duration_days
0.25,6.23,22.00
0.50,7.99,40.50
0.75,14.02,86.25



Median drawdown: 7.99%
Median duration: 40.5 days


### Question 4.  [Stocks] Earnings Surprise Analysis for Amazon (AMZN)


**Calculate the median 2-day percentage change in stock prices following positive earnings surprises days.**

Steps:
1. Load earnings data for the last years using the yfinance method get_earnings_dates():
```python
import yfinance as yf

ticker = 'AMZN'
ticker_obj = yf.Ticker(ticker)
result = ticker_obj.get_earnings_dates()
```
2. Download complete historical price data using yfinance
3. Calculate 2-day percentage changes for all historical dates: for each sequence of 3 consecutive trading days (Day 1, Day 2, Day 3), compute the *return* as Close_Day3 / Close_Day1 - 1. (Assume Day 2 may correspond to the earnings announcement.)
4. What's the correlation of a stock return vs. earnings surprise?

Context: Earnings announcements, especially when they exceed analyst expectations, can significantly impact stock prices in the short term.

Reference: Yahoo Finance earnings calendar - https://finance.yahoo.com/calendar/earnings?symbol=AMZN

### Load earnings data

`get_earnings_dates()` with its default arguments, as the assignment specifies.
The most recent row is a scheduled announcement with no reported EPS or surprise
yet, so rows without a `Surprise(%)` value are dropped.

In [18]:
import yfinance as yf
import pandas as pd

amzn = yf.Ticker('AMZN')
earnings = amzn.get_earnings_dates()

earnings_clean = earnings[earnings['Surprise(%)'].notna()]

print(f'{len(earnings_clean)} reported events, '
      f'{earnings_clean.index.min().date()} → {earnings_clean.index.max().date()}')
print('positive surprises:', (earnings_clean['Surprise(%)'] > 0).sum())
earnings_clean.head()

24 reported events, 2020-10-29 → 2026-07-30
positive surprises: 20


,EPS Estimate,Reported EPS,Surprise(%)
Earnings Date,,,
2026-07-30 16:00:00-04:00,1.83,5.75,215.02
2026-04-29 16:00:00-04:00,1.65,2.78,68.18
2026-02-05 16:00:00-05:00,1.95,1.95,0.22
2025-10-30 16:00:00-04:00,1.56,1.95,25.20
2025-07-31 16:00:00-04:00,1.33,1.68,26.13


### Download complete AMZN price history

Full daily history from the 1997 IPO. `auto_adjust=True` back-adjusts for
stock splits, which matters here: AMZN split 20:1 in June 2022 and three times
in 1998-1999, and on unadjusted close prices each split appears as a one-day
collapse that would contaminate the 2-day returns. AMZN pays no dividend, so
split adjustment is the only adjustment applied.

In [19]:
prices = amzn.history(period='max', auto_adjust=True)
close = prices['Close']

print(close.shape)
print(close.index.min(), '→', close.index.max())
print('NaNs:', close.isna().sum())
print('\nlargest one-day moves (a missed split would show as ~-95%):')
print(close.pct_change().nsmallest(3))

(7377,)
1997-05-15 00:00:00-04:00 → 2026-09-11 00:00:00-04:00
NaNs: 0

largest one-day moves (a missed split would show as ~-95%):
Date
2001-07-24 00:00:00-04:00   -0.247661
2006-07-26 00:00:00-04:00   -0.218220
1998-08-31 00:00:00-04:00   -0.209089
Name: Close, dtype: float64


### Compute 2-day returns for every trading day

For each sequence of three consecutive trading days (Day 1, Day 2, Day 3),
the return is Close(Day 3) / Close(Day 1) - 1, indexed on Day 2. Day 2 is the
day that may carry an earnings announcement, so the window spans the close
before the announcement to the close after it.

`shift(-1)` and `shift(1)` operate on trading-day position rather than calendar
days, so weekends and holidays are handled implicitly. The first and last rows
have no complete window and become NaN.

In [20]:
ret_2d = (close.shift(-1) / close.shift(1) - 1).rename('return_2d')

print('non-null windows:', ret_2d.notna().sum(), 'of', len(ret_2d))
print(f'median across all days: {ret_2d.median():.4%}')

# manual spot check on one date
i = close.index.get_loc(ret_2d.index[100])
print(f'\ncheck {ret_2d.index[100].date()}: '
      f'{close.iloc[i+1]:.4f} / {close.iloc[i-1]:.4f} - 1 = '
      f'{close.iloc[i+1] / close.iloc[i-1] - 1:.6f} vs {ret_2d.iloc[100]:.6f}')

non-null windows: 7375 of 7377
median across all days: 0.1637%

check 1997-10-07: 0.2003 / 0.2062 - 1 = -0.029042 vs -0.029042


### Join earnings to returns and compute the median

The earnings index is tz-aware with a time component (16:00, and occasionally
17:00 or 07:00); the price index is tz-aware at midnight. Both are normalised to
calendar dates before joining.

Announcements that fall on a non-trading day, or after the close such that the
market first reacts the following session, will not match a price index date.
Those are counted rather than silently dropped.

In [21]:
earn = earnings_clean.copy()
earn.index = earn.index.normalize().tz_localize(None)

ret = ret_2d.copy()
ret.index = ret.index.tz_localize(None)

df = earn.join(ret, how='left')

print('events:', len(df), ' matched to a trading day:', df['return_2d'].notna().sum())
print('unmatched:')
print(df[df['return_2d'].isna()].index.date)

positive = df[(df['Surprise(%)'] > 0) & df['return_2d'].notna()]
print(f'\npositive surprises with a return: {len(positive)}')
print(f'median 2-day return: {positive["return_2d"].median():.4%}')

events: 24  matched to a trading day: 24
unmatched:
[]

positive surprises with a return: 20
median 2-day return: 0.3528%


### Correlation of 2-day return vs. earnings surprise

Computed across all 24 events, not just positive surprises, since the question
asks whether returns move with the surprise in either direction. Pearson measures
a linear relationship and is sensitive to extreme surprise percentages; Spearman
ranks instead and is reported alongside for that reason.

In [22]:
matched = df[df['return_2d'].notna()]

print('all events, n =', len(matched))
print(f"  pearson:  {matched['Surprise(%)'].corr(matched['return_2d']):.4f}")
print(f"  spearman: {matched['Surprise(%)'].corr(matched['return_2d'], method='spearman'):.4f}")

print(f"\nsurprise range: {matched['Surprise(%)'].min():.1f}% to {matched['Surprise(%)'].max():.1f}%")

all events, n = 24
  pearson:  0.2217
  spearman: 0.2817

surprise range: -273.3% to 641.9%


### Question 5. [Exploratory, optional] Brainstorm potential idea for your capstone project

I want to build a drawdown risk classifier for the S&P 500 index (^GSPC). For each trading day, the model predicts the probability that the index closes 5% or more below that day's close at some point within the next 21 trading days. On daily data from 1962 to the present, about 16% of days meet this condition, so the class imbalance is mild and the model can be trained on the true class balance, with probability outputs rather than resampling. Features are limited to information available on the prediction day: recent returns, realized volatility, distance from the 52-week high, and market stress indicators. Models are logistic regression as an interpretable baseline and RandomForest, evaluated with walk-forward splits and a 21-day gap to prevent label leakage, scored on precision-recall AUC against the 0.16 base rate, with Brier score and calibration checks. The strategy holds the S&P 500 and reduces exposure when predicted drawdown probability exceeds a threshold, compared against buy-and-hold on CAGR, Sharpe ratio, and maximum drawdown.

### Question 6. [Exploratory, optional] Investigate new metrics

Download three market stress indicators and check their date coverage: VIX from yfinance, and two Treasury/credit spreads from FRED (BAA10Y, T10Y3M), read directly from FRED's CSV endpoint with no API key or extra package.

In [23]:
import pandas as pd
import yfinance as yf

vix = yf.download('^VIX', start='1990-01-01', auto_adjust=False, progress=False)['Close']['^VIX'].rename('VIX')

fred = {s: pd.read_csv(f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={s}',
                       index_col=0, parse_dates=True, na_values='.').iloc[:, 0].rename(s)
        for s in ('BAA10Y', 'T10Y3M')}

for s in (vix, *fred.values()):
    print(s.name, s.first_valid_index().date(), s.last_valid_index().date(), s.notna().sum())

VIX 1990-01-02 2026-09-11 9243
BAA10Y 1986-01-02 2026-09-09 10172
T10Y3M 1982-01-04 2026-09-10 11175


**Coverage (from output above)**
- VIX: [1990-01-02] to [1990-01-02], [9243] observations
- BAA10Y: [1986-01-02] to [2026-09-09], [10172] observations
- T10Y3M: [1982-01-04] to [2026-09-10], [11175] observations

**Why each metric is useful for predicting 21-day drawdowns**

1. **VIX (^VIX):** the market's implied 30-day volatility. Volatility clusters, so when VIX is high, large moves in either direction are more likely over the next month, including declines past 5%.
2. **Baa corporate bond spread over the 10-year Treasury (BAA10Y):** a measure of credit stress. Spreads widen when default risk and financial stress rise, and Gilchrist & Zakrajšek (2012, *American Economic Review*) showed credit spreads carry predictive information about economic activity.
3. **10-year minus 3-month Treasury spread (T10Y3M):** the yield curve slope. Estrella & Mishkin (1998, *Review of Economics and Statistics*) showed curve inversion predicts recessions, and recessions contain the largest equity drawdowns. This is a slow signal, and whether it helps at a 21-day horizon is untested.

**Open:** VIX starts in 1990 and the price data starts in 1962; how to handle the earlier period is a feature-engineering decision for the capstone.